# Import Statements

In [ ]:
import custom_cmap
import os
import sys
from PIL import Image
from IPython.display import display
import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd
from moria import reduce

from pathlib import Path
from astropy.visualization import PercentileInterval

from astropy.io import fits
from astropy.visualization import LogStretch, ImageNormalize
import plotly.express as px
import numpy as np
from astropy.visualization import ImageNormalize, AsinhStretch, SqrtStretch, LogStretch, PowerStretch

plt.rcParams.update({'font.size':25})
plt.rc('text', usetex=True)
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2


# Understanding the main directory. 

The data directory should be organized as follows. You can look at the sample data folder to understand the directory strucutre you need.

<li>00.DATA</li>
<li>01.XYM</li>
<li>02.CMD</li>
<li>03.LOC_TRANS</li>
<li>04.PSF_EXTRACT</li>
<li>05.COORD_TRANS(OPTIONAL)</li>
<li>06.FIT</li>
<li>07.CALIBRATION</li>

All the necessary scripts are included in the sample data directory. The simplest way to run MORIA on your target is to copy the entirety of the "data" folder here, put your exposures in "data/00.DATA", then beginning with the demo.

Using the bright nearby reference stars identified in the CMD, local transformations are derived for each exposure. Pixel data surrounding the target star and the selected PSF-star candidates are extracted from each exposure and accurately transformed into a reference frame corrected for distortion. The extracted pixel list contains the flux location of each pixel relative to the star centers and is fundamental for building the PSF model.

For each filter, MORIA constructs a PSF model using stars selected to have colors and magnitudes similar to those of the target star. Candidate PSF stars are evaluated using residual images generated after PSF subtraction. Stars exhibiting significant residuals—typically caused by blending effects or large Poisson noise—are excluded in a second iteration of the PSF modeling process. The final output is a high-resolution PSF model for each filter.

NOTE: chmod +x program.src is a useful command to use whenever permissions are denied for a script. 

In [ ]:
directory = os.getcwd()

# Step 0 

We assume you ran output_stacks.ipynb and then cmd_diagram.ipynb (in that order) succesfully.

# Step 1

Generate a local coordinate transformation from each exposure frame to the reference frame. This transformation is then used to extract pixels from each exposure and accurately map their positions into the reference frame.

The transformed pixels are subsequently used to construct a PSF model, which is then applied to model the target star.

Note: The step below is computationally intensive and may take a significant amount of time to complete.

In [ ]:
reduce.loc_trans(directory)

# Step 2

Generate a local PSF for each filter using stars with magnitudes and colors similar to those of the target star. Magnitude similarity is particularly important because charge transfer efficiency (CTE) losses can introduce magnitude-dependent variations in the PSF shape. The value entered here should correspond to the number of stars listed in the file "NEARBY_SIM_STARS.XYIVB_targ" located in 02.CMD.

In [ ]:
reduce.extract_psf_1(directory)

# Step 3

Loook at the fit file generated above. The fits file are stored as "show_str_simst.fits" in 04.EXTRACT_PSF/F814W for the F814W filter and "show_str_simstV.fits" in 04.EXTRACT_PSF/F606W for the F606W filter.

To select "good" PSF star candidates, navigate to the bottom row in fits file opened below. Ensure that the residuals are smooth

In [ ]:
#############################
#OPEN FITS FILE AND SCALE IT#
#############################
fit_file = Path(directory).resolve()/f"04.EXTRACT_PSF/F814W/show_str_simst1.fits"
hdul = fits.open(fit_file)
data = hdul[-1].data
hdul.close()
interval = PercentileInterval(90)
scaled = interval(data)

######
#PLOT#
######
fig = px.imshow(scaled, origin='lower', color_continuous_scale='viridis', title="PSF Selection", aspect='equal')
fig.update_layout(width=750, height=750, coloraxis_colorbar=dict(title="Residual", tickvals=[]))
fig.show()

Here are some examples of bad PSF canidates. In these images you can see either black subtractions near the center of the residuals that are not "smooth." 

![bad_psf](bad_psf1.png) ![bad_psf](bad_psf2.png)

You want to select panels that have a smooth residual Here is an example of a good PSF candidate

![good_psf](good_psf.png)

# Step 4: Create a Refined PSF Candidate List

Input the panel number for the PSF stars you like as a list below. The first panel is panel 0:

In [ ]:
good_panels = [3, 4, 11, 14, 15, 16]

In [ ]:
#Change number_of_sim_stars to the number of sim stars you have
number_of_sim_stars=17
good_psf = np.zeros(number_of_sim_stars, dtype = int)
for index in good_panels:
    good_psf[index] = 1

# Step 5: Run PSF Extract again.

This will create a good PSF for your target.

In [ ]:
reduce.extract_psf_2(good_psf, directory)

# Step 6

Loook at the new PSF panels generated after your good selection of stars. 

In [ ]:
#############################
#OPEN FITS FILE AND SCALE IT#
#############################
fit_file = Path(directory).resolve()/f"04.EXTRACT_PSF/F814W/show_str_simst2.fits"
hdul = fits.open(fit_file)
data = hdul[-1].data
hdul.close()
interval = PercentileInterval(90)
scaled = interval(data)

######
#PLOT#
######
fig = px.imshow(scaled, origin='lower', color_continuous_scale='viridis', title="PSF Selection", aspect='equal')
fig.update_layout(width=750, height=750, coloraxis_colorbar=dict(title="Residual", tickvals=[]))
fig.show()

# Step 7

Loook at the actual PSF generated. 

In [ ]:
#############################
#OPEN FITS FILE AND SCALE IT#
#############################
fit_file = Path(directory).resolve()/f"04.EXTRACT_PSF/F814W/psfout_simst2.fits"
hdul = fits.open(fit_file)
data = hdul[-1].data
hdul.close()
interval = PercentileInterval(90)
scaled = interval(data)

######
#PLOT#
######
fig = px.imshow(scaled, origin='lower', color_continuous_scale='viridis', title="PSF Created", aspect='equal')
fig.update_layout(width=750, height=750, coloraxis_colorbar=dict(title="Residual", tickvals=[]))
fig.show()

If the PSF above appears too grainy, run STEP 4-6 again